In [1]:
import numpy 
import torch 
import torch.nn as nn 
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm

### Encoder - Decoder Architecture

In [2]:
class Encoder(nn.Module):

    def __init__(self,input,hidden):
        super().__init__()
        # create a embeddings for the input 
        self.embeddings = nn.Embedding(input,hidden) 
        # Model used for forward pass 
        self.model = nn.GRU(
            input_size=hidden,
            hidden_size=hidden,
            batch_first=True
            )

    def forward(self,X):
        # Convert the input to embeddings 
        X = self.embeddings(X)
        # Forward pass to get me the hidden state (context variable)
        output,hidden = self.model(X)
        # Return the output state
        return output,hidden

class Decoder(nn.Module):

    def __init__(self,vocab_size,hidden):
        super().__init__() 
        # Convert the input to embeddings 
        self.embeddings = nn.Embedding(vocab_size,hidden)
        # GRU decoder 
        self.model = nn.GRU (
            input_size = hidden,
            hidden_size= hidden,
            batch_first=True
        )
        # Forward and construct the output 
        self.fc = nn.Linear(hidden,vocab_size)

    def forward(self,X,context):
        # Convert to embeddings 
        X = self.embeddings(X)
        # decode the input into output using previous context 
        output,hidden = self.model(
            X,context
        )
        # predict the tokens 
        output = self.fc(output)
        # Return the output tokens 
        return output,hidden



### Building Vocab for training the model 

In [3]:
import re
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from collections import Counter

random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Load English-French parallel sentences
dataset = load_dataset("Helsinki-NLP/opus_books", "en-fr")["train"]
dataset = dataset.shuffle(seed=42)
# Small subset for our first experiment
pairs = [
    (row["translation"]["en"], row["translation"]["fr"])
    for row in dataset.select(range(min(10000, len(dataset))))
]

def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower(), re.UNICODE)

# # Reserve special tokens
PAD = "<pad>"
UNK = "<unk>"
BOS = "<bos>"
EOS = "<eos>"

special_tokens = [PAD, UNK, BOS, EOS]

def build_vocab(sentences, min_freq=2):
    counts = Counter()

    for sentence in sentences:
        counts.update(tokenize(sentence))

    vocab = {token: i for i, token in enumerate(special_tokens)}

    for token, count in counts.items():
        if count >= min_freq and token not in vocab:
            vocab[token] = len(vocab)

    return vocab

en_vocab = build_vocab([en for en, fr in pairs])
fr_vocab = build_vocab([fr for en, fr in pairs])

print("English vocabulary:", len(en_vocab))
print("French vocabulary:", len(fr_vocab))
print("Example:", pairs[0])



/home/srikar/miniconda3/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


English vocabulary: 9436
French vocabulary: 10765
Example: ('"Not if it were my own brother!" cried d’Artagnan, as if carried away by his enthusiasm.', "«Non, fût-ce mon frère!» s'écria d'Artagnan comme emporté par l'enthousiasme.")


In [4]:
def encode_sentence(sentence, vocab, add_bos=False):
    tokens = tokenize(sentence)

    ids = [vocab.get(token, vocab[UNK]) for token in tokens]

    if add_bos:
        ids = [vocab[BOS]] + ids

    ids.append(vocab[EOS])
    return ids


class TranslationDataset(Dataset):
    def __init__(self, pairs, en_vocab, fr_vocab):
        self.pairs = pairs
        self.en_vocab = en_vocab    
        self.fr_vocab = fr_vocab

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        en, fr = self.pairs[idx]

        source = encode_sentence(en, self.en_vocab)
        target = encode_sentence(fr, self.fr_vocab, add_bos=True)

        return torch.tensor(source), torch.tensor(target)


def collate_fn(batch):
    sources, targets = zip(*batch)

    sources = nn.utils.rnn.pad_sequence(
        sources, batch_first=True, padding_value=0
    )

    targets = nn.utils.rnn.pad_sequence(
        targets, batch_first=True, padding_value=0
    )

    return sources, targets


data = TranslationDataset(pairs, en_vocab, fr_vocab)

loader = DataLoader(
    data,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

In [5]:
cuda_available = torch.cuda.is_available()
print(f"Is CUDA available?: {cuda_available}")

Is CUDA available?: True


In [8]:
class Seq2Seq(nn.Module):
    def __init__(self, en_vocab_size, fr_vocab_size, hidden):
        super().__init__()

        self.encoder = Encoder(en_vocab_size, hidden)
        self.decoder = Decoder(fr_vocab_size, hidden)

    def forward(self, source, target_input):
        _, context = self.encoder(source)

        logits, _ = self.decoder(target_input, context)

        return logits


hidden = 256

model = Seq2Seq(
    len(en_vocab),
    len(fr_vocab),
    hidden
).to('cpu')

In [9]:
PAD_ID = fr_vocab[PAD]
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.Adam(model.parameters(),lr = 0.001)


epochs = 10


for epoch in tqdm(range(epochs)):

    model.train()

    total_loss = 0 
    total_tokens = 0
    correct_tokens = 0



    for X,y in loader:
        # print("X shape : ",X.shape)
        # print("Y shape : ",y.shape)

        # Teacher Forcing Method 

        target_input = y[:,:-1]
        target_output = y[:,1:]

        optimizer.zero_grad()

        logits = model(X,target_input)

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            target_output.reshape(-1)
                )
        loss.backward()
        # Handling Exploding Gradient
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        predictions = logits.argmax(dim=-1)
        mask = target_output != PAD_ID

        valid_tokens = mask.sum().item()
        correct = ((predictions == target_output) & mask).sum().item()

        total_loss += loss.item() * valid_tokens
        total_tokens += valid_tokens
        correct_tokens += correct
    avg_loss = total_loss / max(total_tokens, 1)
    accuracy = correct_tokens / max(total_tokens, 1)

    print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"Loss: {avg_loss:.4f} "
        f"Token Accuracy: {accuracy:.2%}"
    )



 10%|█         | 1/10 [04:11<37:42, 251.43s/it]

Epoch [1/10] Loss: 5.5876 Token Accuracy: 17.18%


 20%|██        | 2/10 [07:51<31:05, 233.18s/it]

Epoch [2/10] Loss: 4.5658 Token Accuracy: 23.11%


 30%|███       | 3/10 [11:30<26:27, 226.73s/it]

...

Epoch [8/10] Loss: 2.9466 Token Accuracy: 37.17%


 90%|█████████ | 9/10 [32:55<03:34, 214.48s/it]

Epoch [9/10] Loss: 2.7502 Token Accuracy: 40.07%


100%|██████████| 10/10 [3:35:56<00:00, 1295.66s/it]

Epoch [10/10] Loss: 2.5665 Token Accuracy: 43.10%


In [11]:
import torch

def translate(sentence, model, en_vocab, fr_vocab, device, max_len=100):
    model.eval()

    # Tokenize the English input
    tokens = sentence.lower().split()

    # Convert tokens to IDs
    src_ids = [en_vocab["<bos>"]]
    src_ids += [en_vocab[token] for token in tokens]
    src_ids += [en_vocab["<eos>"]]

    print(src_ids)

    source = torch.tensor(
        [src_ids],
        dtype=torch.long,
        device=device
    )

    # Start French decoding with BOS
    bos_id = fr_vocab["<bos>"]
    eos_id = fr_vocab["<eos>"]
    pad_id = fr_vocab["<pad>"]

    target_ids = [bos_id]

    with torch.no_grad():
        for _ in range(max_len):
            target_input = torch.tensor(
                [target_ids],
                dtype=torch.long,
                device=device
            )

            # Predict the next token
            logits = model(source, target_input)

            next_token_logits = logits[0, -1, :].clone()

            # Prevent generating special tokens as ordinary words
            next_token_logits[pad_id] = float("-inf")
            next_token_logits[bos_id] = float("-inf")

            next_id = next_token_logits.argmax().item()

            if next_id == eos_id:
                break

            target_ids.append(next_id)

    # Convert predicted IDs back to French tokens
    id_to_token = {idx: token for token, idx in fr_vocab.items()}

    output_tokens = [
        id_to_token.get(idx, "<UNK>")
        for idx in target_ids[1:]
    ]

    return " ".join(output_tokens)

In [12]:
sentence = "I love you"

translation = translate(
    sentence,
    model,
    en_vocab,
    fr_vocab,
    "cpu"
)

print("English:", sentence)
print("French prediction:", translation)

[2, 107, 700, 127, 3]
English: I love you
French prediction: je ne sais pas .
